# AquaTrack

## STEP 0: PROJECT HEADER
- **Purpose:** apply deterministic preprocessing used by training stages.
- **Dataset:** `dehydration_estimation.csv` / labeled frame.
- **Module:** Main.
- **Inputs:** raw tabular feature frame.
- **Outputs:** imputed, encoded, scaled feature matrix artifacts.

## STEP 1: DATA PREPROCESSING PIPELINE
- Apply deterministic transforms used before modeling.
- **Inputs:** raw/prepared dataframe.
- **Outputs:** imputed, capped, encoded, scaled features.

`models/preprocessing.py` — standard sklearn-style steps (aligned with **`03_target_engineering.ipynb`** primaries):
- **4.1** `SimpleImputer` — **median** (skewed labs); categoricals → **most_frequent**
- **4.2** Optional **IQR** cap / row removal
- **4.3** One-hot / label encoding
- **4.4** **`RobustScaler`** (default for heavy tails); **`StandardScaler`** if features are ~Gaussian (e.g. after log); **`MinMaxScaler`** for bounded features

Run after **01_eda.ipynb**. **`aquatrack_labeled.csv`** adds categorical **`dehydration_level`**.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "models").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

# Legacy package imports removed.
# This notebook should define preprocessing helpers inline.

df = load_prepared()
ID_COL = config.COL_ID
INTERVAL = config.COL_INTERVAL

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.2
CV_FOLDS = 10

## 4.1 Missing value handling

In [ ]:
before = df.isnull().sum().sum()
df_imp = impute_missing(df, numeric_strategy="median", categorical_strategy="most_frequent")
after = df_imp.isnull().sum().sum()
print(f"Total NaN before: {before}, after: {after}")
df_imp.head()

## 4.2 Outlier handling (IQR cap on numeric features, excluding id)

In [ ]:
num_cols, cat_cols = split_column_types(df_imp, id_cols=[ID_COL])
cap_cols = [c for c in num_cols if c != ID_COL]
df_cap = handle_outliers_iqr(df_imp, columns=cap_cols, factor=1.5, mode="cap")
print("IQR cap applied to", len(cap_cols), "numeric columns")

## 4.3 Encoding
- **Label encoding** for `id` (many categories — use only for tree models or drop for linear).
- **Ordinal** treatment: `running interval` is already numeric 0–8; optional one-hot for interval if treating as categorical.

In [ ]:
df_enc, encoders = encode_features(
    df_cap,
    ordinal_cols=None,
    onehot_cols=None,
    label_cols=[ID_COL],
)
print("Encoded label columns:", [k for k in encoders if k.startswith("label_")])
df_enc[[ID_COL, INTERVAL]].head()

## 4.4 Feature scaling
Scale numeric columns (excluding label-encoded id if you prefer — here we scale all numeric except `running interval` optional).

In [ ]:
scale_cols = [c for c in df_enc.select_dtypes(include=[np.number]).columns if c not in (INTERVAL,)]
scale_cols = [c for c in scale_cols if c != INTERVAL]
# RobustScaler matches target_engineering primary pipeline (notebook 03)
df_scaled, scaler = scale_features(df_enc, columns=scale_cols, method="robust")
print(type(scaler).__name__, "fitted on", len(scale_cols), "columns")
df_scaled.iloc[:, :6].describe().T

## Optional: sklearn `ColumnTransformer` pipeline (impute + encode + scale)
Use when you fit a model: drop `id` from features or keep as categorical for one-hot.

In [ ]:
X_raw = df_imp.drop(columns=[ID_COL], errors="ignore")
num_features = [c for c in X_raw.select_dtypes(include=[np.number]).columns if c != INTERVAL]
cat_features = [INTERVAL]  # ordinal discrete
pre = build_sklearn_preprocessor(
    numeric_features=num_features,
    categorical_features=cat_features,
    numeric_impute="median",
    scale="robust",
    one_hot=True,
)
X_mat = pre.fit_transform(X_raw)
print("Transformed shape:", X_mat.shape)